# 04 模型訓練

| | 內容 |
|---|---|
| 輸入 | `data/processed/`（01、03 的輸出） |
| 輸出 | `output/backtest/<時間>_notebook_04_tuning/`（逐窗分數、逐日 6 目標、432 點曲線 `curves.csv`，供 05 讀取）、`output/reports/04_runtimes.json` |

**現行模型**：
- 時刻（佔總分約 93%）：條件經驗分布（日別 × 夏月 × 星期 × 氣溫 2 箱）+ 學習式 LightGBM 在 PMF 層級修正 + 貝氏決策
- 量值：LightGBM 分位數迴歸（τ = 0.625 / 0.70 / 0.50）
- 氣象：五都 4 個溫度項；歷史用 CODiS 觀測，目標日用逐起點校正的 Accuweather
- 太陽光電：Windy 預測發電量 `pv_14`、`pv_sum`
- 6 目標 → 合成 432 點曲線

**回測設計**（`src/evaluation/backtest.py`）：每個窗從起點日 T 的次日 00:00 起算，負載截止於 T 23:50，
預測 3 天共 432 期；目標日的氣象只用預報（honest 模式）。調參組涵蓋全部資料，共 168 窗：
60 個全年起點、2025 與 2026 年 9 月下旬到 10 月上旬的 30 個同季節起點、2026-06-30～09-27 每天一個起點，重複的只算一次。

observed 模式（目標日也用觀測，樂觀）只作對照，可由 `python main.py evaluate --weather-mode observed` 重現。

調參組回測約 35 分鐘。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config").is_dir():   # 找到專案根目錄（notebook 可從任何位置執行）
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import os
import polars as pl
from src.logging_setup import setup_logging

os.chdir(ROOT)
setup_logging(stage="notebook_04", level="WARNING")
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

In [2]:
import datetime as dt
import json
import time

from config import paths, settings
from src import workflow
from src.data import checks
from src.evaluation import backtest

runtimes = {}

## 1. 調參組回測（168 窗）

In [3]:
start = time.perf_counter()
run_dir = backtest.run("tuning", "notebook_04")
runtimes["backtest_tuning"] = round(time.perf_counter() - start, 1)
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
paths.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
{k: summary[k] for k in ("n_windows", "skipped", "manifest_sha256", "data_matches_manifest", "runtime_seconds")}

2026-10-01 15:50:48 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 15:50:49 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 15:50:51 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 15:50:57 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 15:51:00 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


...

2026-10-01 16:41:03 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:07 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:11 | WARNING  | src.workflow | total_score = 0.86197（標準誤 0.04311），最壞折 2.77443（2025-08-01）［氣象模式 honest］
shape: (5, 5)
┌─────────────────┬──────────┬────────┬──────────────┬──────────┐
│ component       ┆ raw      ┆ weight ┆ contribution ┆ share    │
│ ---             ┆ ---      ┆ ---    ┆ ---          ┆ ---      │
│ str             ┆ f64      ┆ f64    ┆ f64          ┆ f64      │
╞═════════════════╪══════════╪════════╪══════════════╪══════════╡
│ s_peak_time     ┆ 5.297886 ┆ 0.15   ┆ 0.794683     ┆ 0.921934 │
│ s_ramp_up       ┆ 0.159252 ┆ 0.15   ┆ 0.023888     ┆ 0.027713 │
│ s_under_penalty ┆ 0.014722 ┆ 1.0    ┆ 0.014722     ┆ 0.017079 │
│ s_ramp_down     ┆ 0.145434 ┆ 0.1    ┆ 0.014543     ┆ 0.016872 │
│ s_peak_mw       ┆ 0.023563 ┆ 0.6    ┆ 0.014138     ┆ 0.016402 │
└─────────────────┴──────────┴────────┴──────────────┴──────────┘


2026-10-01 16:41:11 | WARNING  | src.evaluation.backtest | 回測 [notebook_04／tuning] 168 窗：total_score 0.86197（標準誤 0.04311），同季節 0.66723 → 20261001_155046_notebook_04_tuning


{'n_windows': 168,
 'skipped': [],
 'manifest_sha256': 'ef72d579a4e538d7599be17f778a60dc6d2eb64f0ad6705267b68be07d77f1ef',
 'data_matches_manifest': True,
 'runtime_seconds': 3027.8}

In [4]:
pl.DataFrame([{"範圍": "全體", **summary["overall"]}]
             + [{"範圍": name, **value} for name, value in summary["subsets"].items()])

範圍,n,total_score,stderr,worst,worst_origin,s_peak_mw,s_peak_time,s_ramp_up,s_ramp_down,s_under_penalty
str,i64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""全體""",168,0.861974,0.043114,2.774426,"""2025-08-01""",0.023563,5.297886,0.159252,0.145434,0.014722
"""同季節""",30,0.667229,0.063988,1.777865,"""2025-09-20""",0.022882,4.023056,0.156088,0.143975,0.012231
"""週四五六""",24,0.870988,0.119388,2.37856,"""2025-06-11""",0.024016,5.373226,0.161215,0.13699,0.012714
"""含週六""",74,0.975215,0.064818,2.774426,"""2025-08-01""",0.022945,6.020992,0.189884,0.151772,0.01464
"""夏月末期""",18,0.55494,0.046631,0.888108,"""2025-09-27""",0.022799,3.258755,0.196415,0.163904,0.006594
"""含連假或停班""",37,0.913193,0.078993,1.95883,"""2026-06-19""",0.029527,5.545614,0.228509,0.16849,0.01251


## 2. 以鎖定設定重新訓練並預測提交日（資料截止日次日起 3 天）

與比賽當天 `run_submission.py` 第 5–6 步是同一個函式；下表是每天的 6 個量。

In [5]:
origin = checks.cutoff_date()
days = [origin + dt.timedelta(days=k) for k in range(1, settings.PREDICT_HORIZON_DAYS + 1)]
start = time.perf_counter()
curves, intended, requested = workflow.predict_days(origin, days)
runtimes["train_and_predict"] = round(time.perf_counter() - start, 1)
pl.DataFrame([
    {"日期": str(day), "p_day": t.p_day, "t_day": t.t_day, "p_night": t.p_night,
     "t_night": t.t_night, "ramp_up": t.ramp_up, "ramp_down": t.ramp_down}
    for day, t in intended.items()
])

2026-10-01 16:41:12 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:13 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:15 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:16 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:31 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:35 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:41:40 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


日期,p_day,t_day,p_night,t_night,ramp_up,ramp_down
str,f64,i64,f64,i64,f64,f64
"""2026-10-01""",39873.313879,840,36696.089961,1030,1413.066981,673.009463
"""2026-10-02""",37304.969475,840,35116.704445,1050,1321.752317,678.189983
"""2026-10-03""",32314.733346,840,32414.044778,1090,725.334866,543.416035


In [6]:
(paths.REPORTS_DIR / "04_runtimes.json").write_text(json.dumps(runtimes, indent=2), encoding="utf-8")
runtimes

{'backtest_tuning': 3027.8, 'train_and_predict': 32.6}

## 3. 開發期實驗總表（資料到 2026-06-30，78 窗）

開發期的調參組是 60 個全年起點加上 2025-09-19～10-09 的 21 個同季節起點，重複 3 個，共 78 窗。
每個實驗都走同一套回測（honest 模式），與當時的參考配對比較；當時學習式修正的權重是 0.5。
這些紀錄由 commit `ec3d9a0` 的程式與資料產生，切到該版本即可用同樣的指令重現。
**所有變體都在執行前決定**，看到結果後沒有追加或微調，以免對調參組過度配適。

選模門檻：全部調參窗的配對改善大於 1 個標準誤，且同季節子集沒有變差超過 1 個標準誤；
通過者再以另外 2 個種子確認。差值為正代表變差；「對照」是配對比較的參考紀錄。

In [7]:
EXPERIMENTS = {
    "current_best": "參考：含 UV、不含 Windy",
    "codis_lag1": "CODiS 只到起點前一天",
    "4a_A1_noUV": "氣象不含 UV（採用）",
    "4a_A1raw_noUV_uncal": "目標日預報不校正",
    "4a_B_forecast": "歷史也改用預報",
    "4a_N_noweather": "不用氣象",
    "4c_windy": "加入 Windy 太陽光電預測（採用）",
    "4e_E1_timing_seeds5": "時刻學習模型 5 種子平均",
    "4f_drop_zero_split": "刪除 6 個未使用的特徵（採用）",
    "4g_mix03": "時刻學習權重 0.3",
    "4g_mix07": "時刻學習權重 0.7",
    "4h_magnitude_seeds5": "量值模型 5 種子平均",
    "4d_recursive_ar": "對照：10 分鐘一步的遞迴 AR",
}
names = pl.DataFrame({"label": list(EXPERIMENTS), "實驗": list(EXPERIMENTS.values())})
experiments = names.join(backtest.experiment_table("2026*"), on="label", how="left")
experiments.select("實驗", "total_score", "同季節", "s_peak_time", "差值", "標準誤倍數",
                   "同季節差值", "通過門檻", "對照", "耗時秒")

實驗,total_score,同季節,s_peak_time,差值,標準誤倍數,同季節差值,通過門檻,對照,耗時秒
str,f64,f64,f64,f64,f64,f64,bool,str,f64
"""參考：含 UV、不含 Windy""",1.056294,0.758808,6.580069,null,null,null,null,null,912.1
"""CODiS 只到起點前一天""",1.059357,0.759717,6.600641,0.003063,0.672629,0.000909,false,"""20260927_142000_current_best_tuning""",1482.0
"""氣象不含 UV（採用）""",1.064475,0.760738,6.629968,0.008181,0.807855,0.00193,false,"""20260927_142000_current_best_tuning""",1919.0
"""目標日預報不校正""",1.080949,0.777722,6.744901,0.024655,1.937746,0.018914,false,"""20260927_142000_current_best_tuning""",1975.8
"""歷史也改用預報""",1.076029,0.769114,6.715221,0.019735,1.580864,0.010306,false,"""20260927_142000_current_best_tuning""",2793.4
"""不用氣象""",1.087295,0.799477,6.773392,0.031002,1.911301,0.040669,false,"""20260927_142000_current_best_tuning""",1852.5
"""加入 Windy 太陽光電預測（採用）""",1.041693,0.760246,6.487846,-0.022781,1.810511,-0.000491,true,"""20260928_101300_4a_A1_noUV_tuning""",2097.8
"""時刻學習模型 5 種子平均""",1.059312,0.773529,6.60019,0.003018,0.620476,0.014721,false,"""20260927_142000_current_best_tuning""",4314.5
"""刪除 6 個未使用的特徵（採用）""",1.064475,0.760738,6.629968,0.0,inf,0.0,false,"""20260928_101300_4a_A1_noUV_tuning""",2097.1


In [8]:
experiments.select("實驗", "label", "覆寫")

實驗,label,覆寫
str,str,str
"""參考：含 UV、不含 Windy""","""current_best""","""{}"""
"""CODiS 只到起點前一天""","""codis_lag1""","""{}"""
"""氣象不含 UV（採用）""","""4a_A1_noUV""","""{""WEATHER_METRICS"": [""day_tmax"", ""day_tmin"", ""night_tmax"", ""…"
"""目標日預報不校正""","""4a_A1raw_noUV_uncal""","""{""WEATHER_METRICS"": [""day_tmax"", ""day_tmin"", ""night_tmax"", ""…"
"""歷史也改用預報""","""4a_B_forecast""","""{}"""
"""不用氣象""","""4a_N_noweather""","""{""ENABLE_TIER1_WEATHER"": false, ""TIMING_TEMPERATURE_BINS"": 1…"
"""加入 Windy 太陽光電預測（採用）""","""4c_windy""","""{""ENABLE_WINDY"": true}"""
"""時刻學習模型 5 種子平均""","""4e_E1_timing_seeds5""","""{""TIMING_LEARNED_N_SEEDS"": 5}"""
"""刪除 6 個未使用的特徵（採用）""","""4f_drop_zero_split""","""{""EXCLUDED_FEATURES"": [""is_lunar_new_year_eve"", ""night_price…"


## 4. 以全部資料調參（168 窗）

取得到 2026-09-30 的負載後，只重新調時刻模型中由回測選出的 3 個參數，各試兩個值，在執行前就決定：
學習式修正權重 `TIMING_LEARNED_MIX`（0.5 → 0.3／0.7）、近期同星期分布在 `t_night` 的權重
（0.5 → 0.3／0.7）、`t_night` 的收縮強度（5 → 0／10）。門檻同上，通過者再以 2 個種子確認。

In [9]:
TUNING = {
    "full_mix03": "學習式修正權重 0.3",
    "full_mix07": "學習式修正權重 0.7（採用）",
    "full_recent03": "近期同星期權重 0.3",
    "full_recent07": "近期同星期權重 0.7",
    "full_alpha0": "t_night 收縮強度 0",
    "full_alpha10": "t_night 收縮強度 10",
}
tuning = pl.DataFrame({"label": list(TUNING), "變體": list(TUNING.values())}).join(
    backtest.experiment_table("20261001_*_full_*"), on="label", how="left")
tuning.select("變體", "窗數", "total_score", "同季節", "差值", "標準誤倍數", "同季節差值", "同季節標準誤倍數", "通過門檻")

變體,窗數,total_score,同季節,差值,標準誤倍數,同季節差值,同季節標準誤倍數,通過門檻
str,i64,f64,f64,f64,f64,f64,f64,bool
"""學習式修正權重 0.3""",168,0.899929,0.772512,0.027295,3.092832,0.054409,3.662085,false
"""學習式修正權重 0.7（採用）""",168,0.861974,0.667229,-0.010661,1.144921,-0.050873,2.26406,true
"""近期同星期權重 0.3""",168,0.871949,0.717555,-0.000686,0.286544,-0.000547,0.064757,false
"""近期同星期權重 0.7""",168,0.873716,0.711559,0.001082,0.421747,-0.006544,0.62695,false
"""t_night 收縮強度 0""",168,0.874609,0.719357,0.001974,1.17889,0.001254,0.398919,false
"""t_night 收縮強度 10""",168,0.872012,0.718102,-0.000622,0.885966,0.0,inf,false


In [10]:
# 參考 full_ref 的設定與現行設定只差學習式修正權重（0.5）；三個種子都通過才採用 0.7
backtest.seed_confirmation_table([(42, "full_ref", "full_mix07"),
                                   (1337, "full_ref_seed1337", "full_mix07_seed1337"),
                                   (2718, "full_ref_seed2718", "full_mix07_seed2718")])

種子,參考,候選,差值,標準誤倍數,同季節差值,同季節標準誤倍數,通過門檻
str,f64,f64,f64,f64,f64,f64,bool
"""42""",0.872634,0.861974,-0.010661,1.144921,-0.050873,2.26406,true
"""1337""",0.865058,0.853047,-0.012012,1.269053,-0.054003,2.738415,true
"""2718""",0.86161,0.85179,-0.00982,1.076172,-0.038623,2.104666,true
"""平均""",0.866434,0.855604,-0.010831,1.21185,-0.047833,2.461861,true
